### 1. For cut

The evaluation protocol maintains the same number of trials and the same target/non-target ratio as the original test list. When the original utterances cannot be mapped to valid segments after duration truncation, replacement trials are generated using speaker-aware random sampling to preserve the statistical properties of the evaluation set.

In [4]:
import pandas as pd
import random
from pathlib import Path
from collections import defaultdict
import re

random.seed(42)

In [5]:
def load_metadata(metadata_path):
    df = pd.read_csv(metadata_path)
    seg_map = defaultdict(list)
    speaker_map = {}
    speaker_segments = defaultdict(list)

    for _, row in df.iterrows():
        fname = row["filename"]
        spk = row["speaker_id"]
        
        if re.search(r"\(\d+\)", fname): # skip if filename has (1), (2), ...
            continue
        stem = Path(fname).stem
        original = stem.split("_")[0]

        seg_map[original].append(fname)
        speaker_map[fname] = spk
        speaker_segments[spk].append(fname)

    return seg_map, speaker_map, speaker_segments

def sample_pair(seg_a, seg_b, label, speaker_map, speaker_segments):
    # CASE 1: both A and B
    if seg_a and seg_b:
        for _ in range(10):  # prevent duplicate samples
            sa = random.choice(seg_a)
            sb = random.choice(seg_b)
            if sa != sb:
                return sa, sb
        return None  # fail if duplicated

    # CASE 2: only A have segments
    elif seg_a:
        # label = 1 → one speaker but difference file
        if label == "1":
            if len(seg_a) >= 2:
                sa, sb = random.sample(seg_a, 2)
                return sa, sb
            else:
                return None  # no segment → skip

        # label = 0 → different speaker
        else:
            spk_a = speaker_map[seg_a[0]]
            other_spks = [s for s in speaker_segments if s != spk_a]
            if not other_spks:
                return None
            spk_b = random.choice(other_spks)
            sb = random.choice(speaker_segments[spk_b])
            sa = random.choice(seg_a)
            if sa != sb:
                return sa, sb
            return None

    # CASE 3: only B
    elif seg_b:
        if label == "1":
            if len(seg_b) >= 2:
                sa, sb = random.sample(seg_b, 2)
                return sa, sb
            else:
                return None
        else:
            spk_b = speaker_map[seg_b[0]]

            other_spks = [s for s in speaker_segments if s != spk_b]
            if not other_spks:
                return None

            spk_a = random.choice(other_spks)
            sa = random.choice(speaker_segments[spk_a])
            sb = random.choice(seg_b)

            if sa != sb:
                return sa, sb
            return None

    return None

def generate_test_list(original_test_list, metadata_path, output_path):
    seg_map, speaker_map, speaker_segments = load_metadata(metadata_path)
    # Load trials
    trials = []
    with open(original_test_list) as f:
        for line in f:
            line = line.strip().replace('"','')
            parts = line.split()
            if len(parts) != 3:
                continue
            trials.append(parts)
    target_size = len(trials)

    # keep label ratio
    original_labels = [l for l,_,_ in trials]
    n_pos = original_labels.count("1")
    n_neg = original_labels.count("0")

    pairs = []
    for label, a, b in trials:
        a_id = Path(a).stem
        b_id = Path(b).stem
        seg_a = seg_map.get(a_id, [])
        seg_b = seg_map.get(b_id, [])

        pair = sample_pair(seg_a, seg_b, label, speaker_map, speaker_segments)
        if pair is not None:
            sa, sb = pair
            spk_a = speaker_map[sa]
            spk_b = speaker_map[sb]

            if label == "1" and spk_a != spk_b:
                continue
            if label == "0" and spk_a == spk_b:
                continue

            pairs.append((label, sa, sb))

    # Pool sampling
    speakers = list(speaker_segments.keys())
    while len(pairs) < target_size:
        # keep original ratio
        current_pos = sum(1 for l,_,_ in pairs if l == "1")

        if current_pos < n_pos:
            label = "1"
        else:
            label = "0"

        if label == "1":
            spk = random.choice(speakers)
            segs = speaker_segments[spk]
            if len(segs) < 2:
                continue
            sa, sb = random.sample(segs, 2)
        else:
            spk1, spk2 = random.sample(speakers, 2)
            sa = random.choice(speaker_segments[spk1])
            sb = random.choice(speaker_segments[spk2])

        if sa == sb:
            continue

        pairs.append((label, sa, sb))

    # Final clean
    pairs = pairs[:target_size]

    # Debug
    same_file = sum(1 for _,a,b in pairs if a == b)
    wrong_same = sum(1 for l,a,b in pairs if l=="1" and speaker_map[a]!=speaker_map[b])
    wrong_diff = sum(1 for l,a,b in pairs if l=="0" and speaker_map[a]==speaker_map[b])

    print("Same file pairs:", same_file)
    print("Wrong same-speaker:", wrong_same)
    print("Wrong diff-speaker:", wrong_diff)

    # Save
    with open(output_path, "w") as f:
        for label, a, b in pairs:
            f.write(f"{label}\twav/{a}\twav/{b}\n")

    print("Generated:", len(pairs))

In [ ]:
import os

dir = r"test_data\test_set_O"
durations = [3, 5, 7]

for d in durations:
    meta_path = os.path.join(dir, f"metadata_{d}s.csv")
    test_list_path = os.path.join(dir, "test_list_gt.csv")
    output_path = os.path.join(dir, f"test_list_gt_{d}s.csv")

    generate_test_list(test_list_path, meta_path, output_path)

Same file pairs: 0
Wrong same-speaker: 0
Wrong diff-speaker: 0
Generated: 9895
Same file pairs: 0
Wrong same-speaker: 0
Wrong diff-speaker: 0
Generated: 9895
Same file pairs: 0
Wrong same-speaker: 0
Wrong diff-speaker: 0
Generated: 9895


### 2. For train_vi

In [1]:
import os

def generate_test_list_vi(folder, ori_test_list, out_test_list):
    wav_files = set(
        f for f in os.listdir(folder)
        if f.lower().endswith(".wav")
    )
    print(f"Found {len(wav_files)} wav")

    kept = 0
    removed = 0
    new_lines = []

    with open(ori_test_list, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().replace('"', '')
            parts = line.split()
            if len(parts) != 3:
                continue
            label, a, b = parts
            file_a = os.path.basename(a)
            file_b = os.path.basename(b)

            if file_a in wav_files and file_b in wav_files:
                new_lines.append(f"{label}\twav/{file_a}\twav/{file_b}")
                kept += 1
            else:
                removed += 1

    with open(out_test_list, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines))

    print("-" * 30)
    print("Pairs kept:", kept)
    print("Pairs removed:", removed)

In [ ]:
wav_vi_folder = r"test_data\test_set_O\wav_vi"
original_test_list = r"test_data\test_set_O\test_list_gt.csv"
output_test_list = r"test_data\test_set_O\test_list_gt_vi.csv"

generate_test_list_vi(wav_vi_folder, original_test_list, output_test_list)

Found 16561 wav
------------------------------
Pairs kept: 7148
Pairs removed: 2747


In [7]:
def check_test_list(test_list, wav_folder):
    total = 0
    same = 0
    diff = 0
    missing_files = 0
    duplicate_pairs = 0
    self_pairs = 0
    seen_pairs = set()

    with open(test_list, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip().replace('"', '')
            parts = line.split()
            if len(parts) != 3:
                continue
            label, a, b = parts
            file_a = os.path.basename(a)
            file_b = os.path.basename(b)
            total += 1

            if label == "1":
                same += 1
            else:
                diff += 1

            # check self pair
            if file_a == file_b:
                self_pairs += 1

            # check duplicate
            pair_key = tuple(sorted([file_a, file_b]))
            if pair_key in seen_pairs:
                duplicate_pairs += 1
            else:
                seen_pairs.add(pair_key)

            # check file existence
            path_a = os.path.join(wav_folder, file_a)
            path_b = os.path.join(wav_folder, file_b)
            if not os.path.exists(path_a) or not os.path.exists(path_b):
                missing_files += 1

    print("="*40)
    print("TEST LIST REPORT")
    print("="*40)

    print("Total trials:", total)
    print("Same speaker (1):", same)
    print("Different speaker (0):", diff)

    if total > 0:
        print("Same ratio:", round(same/total, 3))
    print("-"*40)
    print("Missing wav files:", missing_files)
    print("Duplicate pairs:", duplicate_pairs)
    print("Self pairs:", self_pairs)

In [ ]:
test_list = r"test_data\test_set_O\test_list_gt.csv"
wav_folder = r"test_data\test_set_O\wav"
check_test_list(test_list, wav_folder)

TEST LIST REPORT
Total trials: 9895
Same speaker (1): 2446
Different speaker (0): 7449
Same ratio: 0.247
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1002
Self pairs: 0


In [ ]:
test_list_vi = r"test_data\test_set_O\test_list_gt_vi.csv"
wav_folder_vi = r"test_data\test_set_O\wav_vi"
check_test_list(test_list_vi, wav_folder_vi)

TEST LIST REPORT
Total trials: 7148
Same speaker (1): 1151
Different speaker (0): 5997
Same ratio: 0.161
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1002
Self pairs: 0


In [ ]:
durations = [3, 5, 7]
dir = r"test_data\test_set_O"
for d in durations:
    test_list = os.path.join(dir, f"test_list_gt_{d}s.csv")
    wav_folder = os.path.join(dir, f"wav_{d}s")
    check_test_list(test_list, wav_folder)

TEST LIST REPORT
Total trials: 9895
Same speaker (1): 2446
Different speaker (0): 7449
Same ratio: 0.247
----------------------------------------
Missing wav files: 0
Duplicate pairs: 478
Self pairs: 0
TEST LIST REPORT
Total trials: 9895
Same speaker (1): 2446
Different speaker (0): 7449
Same ratio: 0.247
----------------------------------------
Missing wav files: 0
Duplicate pairs: 759
Self pairs: 0
TEST LIST REPORT
Total trials: 9895
Same speaker (1): 2446
Different speaker (0): 7449
Same ratio: 0.247
----------------------------------------
Missing wav files: 0
Duplicate pairs: 1184
Self pairs: 0
